# Scrapping 


## 1. Sumber data

Source : dua halaman daftar eksoplanet di Wikipedia:

- <https://en.wikipedia.org/wiki/List_of_exoplanets_discovered_in_2014>
- <https://en.wikipedia.org/wiki/List_of_exoplanets_discovered_in_2016>

Setiap halaman memuat satu tabel wikitable berisi satu planet per baris. Gabungan kedua tahun menghasilkan 2.367 baris dengan tujuh kolom numerik.

| Kolom | Satuan | Keterangan |
|---|---|---|
| `nama` | - | Nama planet |
| `massa_mj` | massa Jupiter | Massa planet |
| `radius_rj` | radius Jupiter | Radius planet |
| `periode_hari` | hari | Periode orbit |
| `sumbu_semimayor_au` | AU | Sumbu semimayor orbit |
| `suhu_planet_k` | Kelvin | Suhu kesetimbangan planet |
| `metode_deteksi` | - | Metode penemuan (kategorik) |
| `jarak_tahun_cahaya` | tahun cahaya | Jarak dari Bumi |
| `massa_bintang_msun` | massa Matahari | Massa bintang induk |
| `suhu_bintang_k` | Kelvin | Suhu bintang induk |
| `tahun_penemuan` | tahun | 2014 atau 2016 |



In [1]:
%pip install -q beautifulsoup4 requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from importlib.metadata import version
import re
import time
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag

print("beautifulsoup4:", version("beautifulsoup4"))
print("requests      :", version("requests"))
print("pandas        :", version("pandas"))

beautifulsoup4: 4.12.3
requests      : 2.32.3
pandas        : 2.2.3


## 3. Konfigurasi dan pengambilan HTML


In [3]:
BASE_URL = "https://en.wikipedia.org/wiki/List_of_exoplanets_discovered_in_"
TAHUN_TARGET = (2014, 2016)
ALLOWED_LIVE_HOSTS = frozenset({"en.wikipedia.org"})
USER_AGENT = "DM-PracticeBot/1.0 (educational exercise)"
REQUEST_TIMEOUT_SECONDS = 30
MAX_RESPONSE_BYTES = 5_000_000
JEDA_ANTAR_PERMINTAAN_DETIK = 1.0


def validasi_url(url: str) -> None:
    bagian = urlsplit(url)
    if bagian.scheme != "https" or bagian.hostname not in ALLOWED_LIVE_HOSTS:
        raise ValueError("URL harus memakai HTTPS dan host en.wikipedia.org")


def diizinkan_robots(url: str) -> bool:
    bagian = urlsplit(url)
    robots_url = urlunsplit((bagian.scheme, bagian.netloc, "/robots.txt", "", ""))
    try:
        respons = requests.get(
            robots_url,
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        if 400 <= respons.status_code < 500:
            return True
        respons.raise_for_status()
    except requests.RequestException:
        return False

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(respons.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


def ambil_html(url: str) -> str:
    validasi_url(url)
    if not diizinkan_robots(url):
        raise PermissionError("Pengambilan data tidak diizinkan oleh robots.txt")

    respons = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    respons.raise_for_status()
    if "text/html" not in respons.headers.get("Content-Type", "").lower():
        raise ValueError("Respons bukan HTML")
    if len(respons.content) > MAX_RESPONSE_BYTES:
        raise ValueError("Respons melebihi batas ukuran")
    return respons.text

## 4. Ekstraksi dan pembersihan

- `teks_sel()` mengambil teks sel dan membuang penanda rujukan;
- `angka_atau_none()` mengambil bilangan pertama dari teks dan mengembalikan `None` bila tidak ada angka, sehingga sel kosong tidak membuat proses gagal.

In [4]:
POLA_ANGKA = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")
JUMLAH_KOLOM_MINIMAL = 10


def teks_sel(sel: Tag) -> str | None:
    teks = sel.get_text(" ", strip=True)
    teks = re.sub(r"\[.*?\]", "", teks).strip()
    return teks or None


def angka_atau_none(teks: str | None) -> float | None:
    if teks is None:
        return None
    bersih = teks.replace("−", "-").replace(",", "")
    cocok = POLA_ANGKA.search(bersih)
    return float(cocok.group()) if cocok else None


def ekstrak_eksoplanet(dokumen: BeautifulSoup, tahun: int) -> tuple[dict[str, object], ...]:
    tabel = dokumen.select_one("table.wikitable")
    if tabel is None:
        return ()

    def ekstrak_satu(baris: Tag) -> dict[str, object] | None:
        sel = baris.find_all(["td", "th"])
        if len(sel) < JUMLAH_KOLOM_MINIMAL:
            return None
        nilai = [teks_sel(satu_sel) for satu_sel in sel[:JUMLAH_KOLOM_MINIMAL]]
        if nilai[0] is None:
            return None
        return {
            "nama": nilai[0],
            "massa_mj": angka_atau_none(nilai[1]),
            "radius_rj": angka_atau_none(nilai[2]),
            "periode_hari": angka_atau_none(nilai[3]),
            "sumbu_semimayor_au": angka_atau_none(nilai[4]),
            "suhu_planet_k": angka_atau_none(nilai[5]),
            "metode_deteksi": nilai[6],
            "jarak_tahun_cahaya": angka_atau_none(nilai[7]),
            "massa_bintang_msun": angka_atau_none(nilai[8]),
            "suhu_bintang_k": angka_atau_none(nilai[9]),
            "tahun_penemuan": tahun,
        }

    hasil = (ekstrak_satu(baris) for baris in tabel.select("tr")[1:])
    return tuple(baris for baris in hasil if baris is not None)

## 5. Scraping 


In [5]:
semua_baris: list[dict[str, object]] = []

for urutan, tahun in enumerate(TAHUN_TARGET):
    if urutan:
        time.sleep(JEDA_ANTAR_PERMINTAAN_DETIK)

    url = f"{BASE_URL}{tahun}"
    try:
        dokumen = BeautifulSoup(ambil_html(url), "html.parser")
        baris_tahun = ekstrak_eksoplanet(dokumen, tahun)
        semua_baris.extend(baris_tahun)
        print(f"{tahun}: {len(baris_tahun)} baris diambil")
    except (requests.RequestException, PermissionError, ValueError) as error:
        print(f"{tahun}: dilewati dengan aman ({error})")

df_eksoplanet = pd.DataFrame(semua_baris)
print("Ukuran data:", df_eksoplanet.shape)
df_eksoplanet.head(10)

2014: 869 baris diambil
2016: 1498 baris diambil
Ukuran data: (2367, 11)


,nama,massa_mj,radius_rj,periode_hari,sumbu_semimayor_au,suhu_planet_k,metode_deteksi,jarak_tahun_cahaya,massa_bintang_msun,suhu_bintang_k,tahun_penemuan
0,BD-11 4672 b,0.5300,NaN,1667.00000,2.2800,NaN,radial vel.,89.00,0.57,4475.0,2014
1,Beta Cancri b,7.8000,NaN,605.20000,1.7000,NaN,radial vel.,290.40,1.70,4092.1,2014
2,Beta Ursae Minoris b,6.1000,NaN,522.30000,1.4000,NaN,radial vel.,126.50,1.40,4126.0,2014
3,COROT-22b,0.0380,0.435,9.75598,0.0920,885.0,transit,1930.00,1.10,5939.0,2014
4,COROT-24b,0.0180,0.330,5.11340,0.0560,1070.0,transit,2000.00,0.91,4950.0,2014
5,COROT-24c,0.0880,0.440,11.75900,0.0980,850.0,transit,2000.00,0.91,4950.0,2014
6,COROT-27b,10.3900,1.007,3.57532,0.0476,1500.0,transit,3400.00,1.05,5900.0,2014
7,Gliese 15 Ab,0.0170,NaN,11.44330,0.0717,NaN,radial vel.,11.60,0.38,3567.0,2014
8,Gliese 27.1 b,0.0409,NaN,15.81900,0.1010,NaN,radial vel.,77.05,0.53,3542.0,2014
9,Gliese 180 b,0.0261,NaN,17.38000,0.1030,NaN,radial vel.,38.95,0.43,3371.0,2014


## 6. Validasi hasil

Data diperiksa terhadap jumlah baris, keunikan nama planet, dan kewajaran nilai numerik.

In [6]:
assert len(df_eksoplanet) >= 1500, "Jumlah baris kurang dari 1500"
assert df_eksoplanet["nama"].is_unique, "Nama planet harus unik"
assert df_eksoplanet["periode_hari"].dropna().gt(0).all(), "Periode orbit harus positif"
assert df_eksoplanet["jarak_tahun_cahaya"].dropna().gt(0).all(), "Jarak harus positif"

print("Jumlah baris :", len(df_eksoplanet))
print("Jumlah kolom :", df_eksoplanet.shape[1])
print()
print("Persentase nilai terisi per kolom:")
print((df_eksoplanet.notna().mean() * 100).round(1).to_string())
print()
print("Sebaran metode deteksi:")
print(df_eksoplanet["metode_deteksi"].value_counts().to_string())

Jumlah baris : 2367
Jumlah kolom : 11

Persentase nilai terisi per kolom:
nama                  100.0
massa_mj               12.6
radius_rj              94.6
periode_hari           99.2
sumbu_semimayor_au     43.0
suhu_planet_k           4.4
metode_deteksi        100.0
jarak_tahun_cahaya     96.8
massa_bintang_msun     88.8
suhu_bintang_k         98.9
tahun_penemuan        100.0

Sebaran metode deteksi:
metode_deteksi
transit         2235
radial vel.      105
microlensing      13
timing             9
imaging            5


In [7]:
df_eksoplanet.describe().round(3)

,massa_mj,radius_rj,periode_hari,sumbu_semimayor_au,suhu_planet_k,jarak_tahun_cahaya,massa_bintang_msun,suhu_bintang_k,tahun_penemuan
count,299.000,2240.000,2348.000,1018.000,104.000,2291.000,2101.000,2340.000,2367.000
mean,2.073,0.263,77.301,9.352,1232.308,2470.103,0.955,5488.815,2015.266
std,3.632,0.276,459.522,225.202,534.783,1707.256,0.239,669.083,0.964
min,0.000,0.042,0.538,0.011,131.000,4.200,0.080,2559.000,2014.000
25%,0.085,0.132,4.634,0.057,822.750,1437.000,0.840,5178.000,2014.000
50%,0.729,0.189,11.039,0.098,1292.000,2320.000,0.950,5636.500,2016.000
75%,2.151,0.255,26.681,0.182,1604.250,3230.000,1.070,5935.000,2016.000
max,30.000,3.400,11613.000,6900.000,2508.000,27000.000,3.210,10500.000,2016.000


### Data Result

Saved as csv format

In [8]:
OUTPUT_PATH = Path("eksoplanet.csv")
df_eksoplanet.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"{len(df_eksoplanet)} baris disimpan ke {OUTPUT_PATH.resolve()}")

2367 baris disimpan ke /Users/elyne/Desktop/AI Journey/ITS/DaMin/DataMining-5054251011-Maria-Putu-Evelyne-Corena-Puryatma-ModTask/Task 1 - EDA and Scrapping/Tugas/eksoplanet.csv
